In [1]:
import os

video_path = '../demo/demo_helmet.mp4' 
if not os.path.exists(video_path):
    print(f"No path: '{video_path}' ")
else:
    print("Yes path")

No path: '../demo/demo_helmet.mp4' 


In [2]:
import os
import cv2
from ultralytics import YOLO

video_path = '../demo/demo3.mp4' 

if not os.path.exists(video_path):
    print(f"Error: Video file not found at '{video_path}'")
else:
    print(f"Video found: '{video_path}'")
    
    # Load model
    model_path = 'runs/detect/runs/detect/train_helmet/weights/best.pt'
    model = YOLO(model_path)

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error: OpenCV cannot open the video file.")
    else:
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = int(cap.get(cv2.CAP_PROP_FPS))

        out_name = 'demo_helmet_guard.mp4'
        out = cv2.VideoWriter(out_name, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

        print("Processing video, please wait...")

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: 
                break

            results = model(frame, verbose=False)
            result = results[0]

            persons = []
            helmets = []

            for box in result.boxes:
                cls_id = int(box.cls[0])
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                
                if cls_id == 1:
                    persons.append((x1, y1, x2, y2))
                elif cls_id == 0:
                    helmets.append((x1, y1, x2, y2))

            for (px1, py1, px2, py2) in persons:
                head_y_bottom = py1 + (py2 - py1) // 3
                has_helmet = False
                
                for (hx1, hy1, hx2, hy2) in helmets:
                    if (px1 < hx2 and px2 > hx1 and py1 < hy2 and head_y_bottom > hy1):
                        has_helmet = True
                        cv2.rectangle(frame, (hx1, hy1), (hx2, hy2), (0, 0, 255), 2)
                        cv2.putText(frame, 'Helmet', (hx1, hy1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
                        break
                        
                if has_helmet:
                    cv2.rectangle(frame, (px1, py1), (px2, py2), (0, 255, 0), 2)
                else:
                    cv2.rectangle(frame, (px1, py1), (px2, py2), (255, 0, 0), 3)
                    cv2.putText(frame, 'NO_HELMET', (px1, py1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

            out.write(frame)

        cap.release()
        out.release()
        
        save_path = os.path.abspath(out_name)
        print("Video exported successfully!")
        print(f"Saved at: {save_path}")

Video found: '../demo/demo3.mp4'
Processing video, please wait...
Video exported successfully!
Saved at: /home/admin/Documents/HelmetGuard/notebooks/demo_helmet_guard.mp4
